# T5-Small Fine-Tuning on CNN/DailyMail (Summarization)

Run cells top to bottom. **Runtime > Change runtime type > GPU (T4)** before starting.

If you re-run the install cell after already importing `transformers`, restart the runtime (**Runtime > Restart session**) before continuing, or the newer `eval_strategy` / `processing_class` arguments below may fail.

**Note on the dataset:** the old bare `cnn_dailymail` repo relied on a loading script, which newer `datasets` versions (4.0+) refuse to execute — the same class of error you hit with AG News and CoNLL-2003. This notebook uses `abisee/cnn_dailymail`, the official parquet-converted mirror with the same `article`/`highlights`/`id` fields and the same `1.0.0`/`2.0.0`/`3.0.0` configs.

**Heads-up on runtime:** CNN/DailyMail has ~287k training articles. Full T5 fine-tuning for 3 epochs on a free-tier Colab T4 can take several hours and may outlast the session. A commented-out subsampling line is included in the config cell if you want to test the pipeline on a smaller slice first.

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate sentencepiece rouge_score

## Restart runtime here if this is not a fresh session
`Runtime > Restart session`, then continue from the next cell.

In [ ]:
import os
import numpy as np
import torch
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

set_seed(42)

## Environment check

In [ ]:
print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")
    print("Training will be significantly slower.")
    print("Go to Runtime > Change runtime type > GPU.")

print("=" * 70)

## Configuration

In [ ]:
MODEL_NAME = "t5-small"

DATASET_NAME = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"

OUTPUT_DIR = "./summarization_results"
FINAL_MODEL_DIR = "./best_t5_summarizer"

# Maximum length of input article
MAX_INPUT_LENGTH = 512

# Maximum length of generated summary
MAX_TARGET_LENGTH = 128

LEARNING_RATE = 5e-5

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8

NUM_EPOCHS = 3

WEIGHT_DECAY = 0.01

USE_FP16 = torch.cuda.is_available()

# If you hit an out-of-memory error on a free-tier Colab GPU, lower this
# TRAIN_BATCH_SIZE = 4

# For a quick pipeline test instead of the full ~287k-article train set,
# subsample after loading the dataset in the next cell, e.g.:
# dataset["train"] = dataset["train"].shuffle(seed=42).select(range(20000))

## Load CNN/DailyMail dataset

In [ ]:
print("\n" + "=" * 70)
print("LOADING CNN/DAILYMAIL DATASET")
print("=" * 70)

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG
)

print(dataset)

print("\nDataset sizes:")

for split in dataset:
    print(f"{split}: {len(dataset[split])}")

## Load T5 tokenizer

In [ ]:
print("\n" + "=" * 70)
print("LOADING T5 TOKENIZER")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")

## Preprocess data
T5 works using a text-to-text format. We add a task prefix to tell T5 what we want it to do.

In [ ]:
print("\n" + "=" * 70)
print("PREPARING DATASET")
print("=" * 70)

PREFIX = "summarize: "


def preprocess_function(examples):

    inputs = [
        PREFIX + article
        for article in examples["article"]
    ]

    targets = examples["highlights"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Dataset preprocessing completed.")

## Load pretrained T5 model

In [ ]:
print("\n" + "=" * 70)
print("LOADING T5-SMALL")
print("=" * 70)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("T5-Small loaded successfully.")

## ROUGE evaluation metric

In [ ]:
print("\n" + "=" * 70)
print("SETTING UP ROUGE EVALUATION")
print("=" * 70)

rouge_metric = evaluate.load(
    "rouge"
)


def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    # Replace -100 with the tokenizer's padding token
    # so we can decode the labels.
    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    # Strip unnecessary whitespace
    decoded_predictions = [
        prediction.strip()
        for prediction in decoded_predictions
    ]

    decoded_labels = [
        label.strip()
        for label in decoded_labels
    ]

    results = rouge_metric.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": results["rouge1"],
        "rouge2": results["rouge2"],
        "rougeL": results["rougeL"]
    }


print("ROUGE evaluation ready.")

## Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

## Training configuration

In [ ]:
print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

training_args = Seq2SeqTrainingArguments(

    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=WEIGHT_DECAY,

    predict_with_generate=True,

    generation_max_length=MAX_TARGET_LENGTH,

    load_best_model_at_end=True,

    metric_for_best_model="rougeL",

    greater_is_better=True,

    fp16=USE_FP16,

    save_total_limit=2,

    logging_steps=100,

    report_to="none"
)

## Create trainer

In [ ]:
print("\n" + "=" * 70)
print("CREATING SEQUENCE-TO-SEQUENCE TRAINER")
print("=" * 70)

trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_datasets["train"],

    eval_dataset=tokenized_datasets["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Trainer ready.")

## Train T5

In [ ]:
print("\n" + "=" * 70)
print("STARTING T5 FINE-TUNING")
print("=" * 70)

trainer.train()

print("\nTraining completed successfully!")

## Final evaluation

In [ ]:
print("\n" + "=" * 70)
print("FINAL MODEL EVALUATION")
print("=" * 70)

evaluation_results = trainer.evaluate(
    max_length=MAX_TARGET_LENGTH
)

for key, value in evaluation_results.items():

    if isinstance(value, float):

        print(
            f"{key}: {value:.4f}"
        )

    else:

        print(
            f"{key}: {value}"
        )

## Save best model

In [ ]:
print("\n" + "=" * 70)
print("SAVING FINAL SUMMARIZATION MODEL")
print("=" * 70)

os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print("Model saved successfully!")

print(
    "Model location:",
    os.path.abspath(FINAL_MODEL_DIR)
)

## Verify saved model files

In [ ]:
print("\n" + "=" * 70)
print("SAVED MODEL FILES")
print("=" * 70)

for filename in sorted(
    os.listdir(FINAL_MODEL_DIR)
):

    filepath = os.path.join(
        FINAL_MODEL_DIR,
        filename
    )

    if os.path.isfile(filepath):

        size_mb = (
            os.path.getsize(filepath)
            / (1024 * 1024)
        )

        print(
            f"{filename:<40}"
            f"{size_mb:.2f} MB"
        )

## Reload saved model

In [ ]:
print("\n" + "=" * 70)
print("RELOADING SAVED MODEL")
print("=" * 70)

trained_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

trained_model = AutoModelForSeq2SeqLM.from_pretrained(
    FINAL_MODEL_DIR
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

trained_model.to(device)

trained_model.eval()

print("Saved T5 model successfully reloaded.")
print("Running on:", device)

## Summarization function

In [ ]:
def summarize_text(
    text,
    max_length=128,
    min_length=20
):

    input_text = PREFIX + text

    inputs = trained_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        summary_ids = trained_model.generate(

            **inputs,

            max_length=max_length,

            min_length=min_length,

            num_beams=4,

            length_penalty=1.0,

            early_stopping=True
        )

    summary = trained_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

## Test summarization

In [ ]:
print("\n" + "=" * 70)
print("TESTING FINE-TUNED SUMMARIZATION MODEL")
print("=" * 70)

test_article = """
Artificial intelligence has rapidly transformed many industries
over the past decade. Companies are increasingly using machine
learning systems to automate repetitive tasks, analyze large
amounts of data, and improve decision making. Recent advances in
large language models have also made it possible for computers
to understand and generate human language with remarkable
accuracy. However, researchers warn that organizations need to
consider issues such as privacy, security, bias, and the impact
of automation on employment as these technologies continue to
develop.
"""


summary = summarize_text(
    test_article
)

print("\nORIGINAL TEXT:")
print(test_article)

print("\nGENERATED SUMMARY:")
print(summary)

## Second test

In [ ]:
print("\n" + "=" * 70)
print("SECOND SUMMARIZATION TEST")
print("=" * 70)

second_article = """
Scientists have developed a new battery technology that could
significantly improve the range and charging speed of electric
vehicles. The researchers say the new material allows batteries
to store more energy while reducing charging time. Several
automotive companies have expressed interest in the technology,
although large-scale manufacturing and long-term durability
tests are still required before it can be used in commercial
vehicles.
"""

second_summary = summarize_text(
    second_article
)

print("\nORIGINAL TEXT:")
print(second_article)

print("\nGENERATED SUMMARY:")
print(second_summary)

## Summary

In [ ]:
print("\n" + "=" * 70)
print("TEXT SUMMARIZATION MODEL COMPLETE")
print("=" * 70)

print("Task: Text Summarization")
print("Base Model:", MODEL_NAME)
print("Dataset: CNN/DailyMail")
print("Fine-tuned Model:", FINAL_MODEL_DIR)

print("\nStatus:")
print("Trained       ✓")
print("Evaluated     ✓")
print("Saved         ✓")
print("Reloaded      ✓")
print("Tested        ✓")

print("=" * 70)